[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/12.%20The%20Berth%20Allocation%20with%20Tidal%20Windows%20Problem/P12-Tier-3_executed.ipynb)

---


# 12. The Berth Allocation with Tidal Windows Problem
## Tier 3 — Cuckoo Search Metaheuristic

### Goal

Use Cuckoo Search to find a low-cost feasible schedule for vessels competing for berths while respecting arrival times, draft requirements, tidal windows, and berth non-overlap. The method is stochastic: it explores a population of candidate schedules and does not guarantee that the global optimum will be found.

### Inputs and outputs

The input consists of vessel characteristics, berth depths, high-tide windows, and search parameters. A candidate schedule assigns each vessel to a berth and a service start time. The output is the best schedule found, its objective value, and a history of the best objective over the search iterations.

### Modeling notes

A tidal window is treated as an allowable **service start-time window**. The berth must provide enough depth when service starts; service may continue after the window closes. This interpretation makes the given three-hour tidal windows compatible with the vessels' longer processing times. Start times are searched on a 0.5-hour grid. Each vessel remains on its assigned berth for its full processing time, and two vessels may not occupy the same berth at overlapping times. Berth length, vessel length, channel access, setup time, weather, and uncertainty in tide or processing duration are outside this example.

The objective uses the vessel's demurrage rate multiplied by its elapsed time from arrival until service completion. Thus, the model minimizes a weighted completion-delay cost, not simply the number of hours vessels wait before service starts.

### Cuckoo Search in this problem

A *nest* represents one complete schedule. A new candidate, or *egg*, is generated by changing one vessel's berth or start time. Lévy-flight steps provide a heavy-tailed random signal: most steps are modest, while occasional large steps encourage broader exploration. The discrete berth and time choices translate that signal into changes in the schedule. Candidates compete for survival, and some low-fitness nests are abandoned and regenerated.

The search follows this sequence:

1. Generate a population of schedules.
2. Generate and evaluate candidate schedules through Lévy-flight-guided changes.
3. Keep improved candidates and update the best schedule found.
4. With the discovery probability, replace a fraction of the weakest nests.
5. Record the best objective and repeat.

The method balances exploration and exploitation but is a metaheuristic, not a proof of optimality. Its results can vary between runs. The sensitivity experiments later in the notebook examine how selected search parameters affect observed outcomes; they do not establish universally optimal settings.

## 1. Imports and reproducibility

The notebook uses NumPy for numerical operations, Matplotlib for plots, and Python's `random` module for discrete choices. Both random-number generators are seeded so that a run is reproducible in the same software environment. Execution time can still vary across machines.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import copy
import math
import random
import time

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import gamma

np.random.seed(42)
random.seed(42)

## 2. Representing vessels, berths, tidal windows, and schedules

A data class groups the attributes that belong together. Vessel and berth identifiers are used as list indices in this example, so the supplied identifiers start at zero and are consecutive. The `Nest` class stores a complete candidate schedule and its evaluated quality.

In [ ]:
@dataclass
class Vessel:
    """A vessel and the service requirements used by the model."""
    id: int
    draft: float
    processing_time: float
    arrival_time: float
    demurrage_cost: float


@dataclass
class Berth:
    """A berth with low-tide and high-tide depth limits."""
    id: int
    length: float
    depth_low_tide: float
    depth_high_tide: float


@dataclass
class TidalWindow:
    """A time interval in which service may start at high-tide depth."""
    start_time: float
    end_time: float
    depth_factor: float


@dataclass
class Nest:
    """A candidate schedule and its objective, feasibility, and fitness."""
    vessel_assignments: Dict[int, int]
    start_times: Dict[int, float]
    objective_value: float
    feasible: bool
    fitness: float

## 3. Concrete example

The example contains three vessels, two berths, and two high-tide windows over a 24-hour planning period. Depths are measured in metres, and processing and arrival times are measured in hours. The demurrage rate is the objective-cost weight per hour of elapsed time after arrival.

In [ ]:
vessels = [
    Vessel(id=0, draft=12.0, processing_time=4.0, arrival_time=6.0, demurrage_cost=8000),
    Vessel(id=1, draft=14.0, processing_time=6.0, arrival_time=8.0, demurrage_cost=12000),
    Vessel(id=2, draft=11.0, processing_time=3.0, arrival_time=14.0, demurrage_cost=6000),
]

berths = [
    Berth(id=0, length=300.0, depth_low_tide=10.0, depth_high_tide=15.0),
    Berth(id=1, length=350.0, depth_low_tide=11.0, depth_high_tide=16.0),
]

tidal_windows = [
    TidalWindow(start_time=6.5, end_time=9.5, depth_factor=1.0),
    TidalWindow(start_time=19.0, end_time=22.0, depth_factor=1.0),
]

print(f"Problem setup: {len(vessels)} vessels, {len(berths)} berths, {len(tidal_windows)} tidal windows")
print("Vessels (id, draft, processing time, arrival time):")
for vessel in vessels:
    print(f"  ({vessel.id}, {vessel.draft}, {vessel.processing_time}, {vessel.arrival_time})")
print("Berths (id, low-tide depth, high-tide depth):")
for berth in berths:
    print(f"  ({berth.id}, {berth.depth_low_tide}, {berth.depth_high_tide})")

## 4. Feasibility and objective functions

A schedule is feasible when every vessel starts no earlier than its arrival, starts during a tidal window, has sufficient depth at its assigned berth at the start time, and does not overlap another vessel on that berth. Overlap is tested on each berth independently; simultaneous service at different berths is allowed.

For vessel \(i\), let \(s_i\) be its service start time, \(p_i\) its processing time, \(a_i\) its arrival time, and \(c_i\) its demurrage rate. Its completion time is \(s_i+p_i\), and its contribution to the objective is \(c_i(s_i+p_i-a_i)\). The total objective is

$$
\min \quad \sum_{i \in V} c_i(s_i+p_i-a_i)
$$

subject to

$$
s_i \geq a_i, \qquad s_i \in W, \qquad d_{b_i}(s_i) \geq r_i,
$$

and, for vessels \(i\) and \(j\) assigned to the same berth,

$$
s_i+p_i \leq s_j \quad\text{or}\quad s_j+p_j \leq s_i.
$$

Here \(V\) is the vessel set, \(W\) is the union of tidal start-time windows, \(b_i\) is vessel \(i\)'s assigned berth, \(d_{b_i}(s_i)\) is the available depth at that berth at the start time, and \(r_i\) is the vessel's draft. The disjunction means that one of the two vessels must finish before the other starts. In the code, `is_feasible_assignment` checks the individual conditions, `is_nest_feasible` checks all vessel assignments and berth overlaps, and `calculate_objective` evaluates the sum. An infeasible candidate receives zero fitness so that feasible schedules are preferred.

In [ ]:
def get_tidal_depth(time: float, berth: Berth) -> float:
    """Return the berth depth available at a proposed service start time."""
    for window in tidal_windows:
        if window.start_time <= time <= window.end_time:
            return berth.depth_high_tide
    return berth.depth_low_tide


def is_feasible_assignment(vessel: Vessel, berth: Berth, start_time: float) -> bool:
    """Check arrival, tidal-window, and draft requirements for one vessel."""
    if start_time < vessel.arrival_time:
        return False

    in_tidal_window = any(
        window.start_time <= start_time <= window.end_time
        for window in tidal_windows
    )
    if not in_tidal_window:
        return False

    return get_tidal_depth(start_time, berth) >= vessel.draft


def calculate_objective(nest: Nest) -> float:
    """Return the total weighted time from vessel arrival to completion."""
    total_cost = 0.0
    for vessel_id in nest.vessel_assignments:
        vessel = vessels[vessel_id]
        completion_time = nest.start_times[vessel_id] + vessel.processing_time
        total_cost += vessel.demurrage_cost * (completion_time - vessel.arrival_time)
    return total_cost


def is_nest_feasible(nest: Nest) -> bool:
    """Check every assignment and all same-berth non-overlap constraints."""
    if set(nest.vessel_assignments) != {vessel.id for vessel in vessels}:
        return False
    if set(nest.start_times) != {vessel.id for vessel in vessels}:
        return False

    berth_schedules: Dict[int, List[Tuple[float, float, int]]] = {}
    for vessel_id, berth_id in nest.vessel_assignments.items():
        if not 0 <= berth_id < len(berths):
            return False

        vessel = vessels[vessel_id]
        start_time = nest.start_times[vessel_id]
        berth = berths[berth_id]
        if not is_feasible_assignment(vessel, berth, start_time):
            return False

        end_time = start_time + vessel.processing_time
        berth_schedules.setdefault(berth_id, []).append(
            (start_time, end_time, vessel_id)
        )

    for schedule in berth_schedules.values():
        schedule.sort()
        for current, following in zip(schedule, schedule[1:]):
            if current[1] > following[0]:
                return False

    return True


def calculate_fitness(nest: Nest) -> float:
    """Convert a minimization objective into a fitness to maximize."""
    if not nest.feasible or not math.isfinite(nest.objective_value):
        return 0.0
    return 1.0 / (1.0 + nest.objective_value)


print("Feasibility and objective functions are ready.")

## 5. Generating initial nests

A population gives the search several starting schedules rather than a single starting point. The generator randomizes vessel order, berth assignments, and feasible start times, then checks whether the resulting schedule has berth conflicts. It retries because individually valid vessel assignments can still conflict when combined. If it cannot construct a feasible schedule within the attempt limit, it returns an explicitly infeasible candidate with zero fitness rather than labeling an invalid schedule as feasible.

In [ ]:
def generate_random_nest() -> Nest:
    """Generate a randomized feasible schedule when one is found."""
    max_attempts = 1000
    time_step = 0.5

    for _ in range(max_attempts):
        vessel_order = vessels.copy()
        random.shuffle(vessel_order)
        vessel_assignments: Dict[int, int] = {}
        start_times: Dict[int, float] = {}
        berth_schedules: Dict[int, List[Tuple[float, float]]] = {}
        feasible = True

        for vessel in vessel_order:
            berth_id = random.randrange(len(berths))
            berth = berths[berth_id]
            candidate_times = np.arange(vessel.arrival_time, 24.0, time_step)
            feasible_times = []

            for start_time in candidate_times:
                if not is_feasible_assignment(vessel, berth, float(start_time)):
                    continue

                end_time = float(start_time) + vessel.processing_time
                overlaps = any(
                    float(start_time) < existing_end
                    and existing_start < end_time
                    for existing_start, existing_end in berth_schedules.get(berth_id, [])
                )
                if not overlaps:
                    feasible_times.append(float(start_time))

            if not feasible_times:
                feasible = False
                break

            start_time = random.choice(feasible_times)
            vessel_assignments[vessel.id] = berth_id
            start_times[vessel.id] = start_time
            berth_schedules.setdefault(berth_id, []).append(
                (start_time, start_time + vessel.processing_time)
            )

        if feasible:
            nest = Nest(
                vessel_assignments=vessel_assignments,
                start_times=start_times,
                objective_value=0.0,
                feasible=False,
                fitness=0.0,
            )
            nest.feasible = is_nest_feasible(nest)
            nest.objective_value = calculate_objective(nest)
            nest.fitness = calculate_fitness(nest)
            if nest.feasible:
                return nest

    fallback = Nest(
        vessel_assignments={vessel.id: vessel.id % len(berths) for vessel in vessels},
        start_times={vessel.id: vessel.arrival_time for vessel in vessels},
        objective_value=float("inf"),
        feasible=False,
        fitness=0.0,
    )
    return fallback


print("Testing randomized nest generation:")
for index in range(5):
    nest = generate_random_nest()
    print(
        f"Nest {index + 1}: Feasible={nest.feasible}, "
        f"Objective={nest.objective_value:.2f}, Fitness={nest.fitness:.8f}"
    )

## 6. Lévy flights and Cuckoo Search operators

A Lévy-flight step is drawn using Mantegna's construction. For a parameter \(0<\beta<2\), independent standard normal random variables \(u\) and \(v\) give

$$
\sigma_u = \left[\frac{\Gamma(1+\beta)\sin(\pi\beta/2)}
{\Gamma((1+\beta)/2)\,\beta\,2^{(\beta-1)/2}}\right]^{1/\beta},
\qquad
L = \frac{u\sigma_u}{|v|^{1/\beta}}.
$$

The heavy-tailed step \(L\) is a search signal, not a berth coordinate or a time measured in hours. The implementation maps its magnitude to discrete schedule changes. Small steps favor nearby start times and the best-known schedule; larger steps permit broader random changes. A proposed change can violate berth non-overlap, so it is evaluated by the same feasibility function as every other nest. The abandonment operator replaces the weakest fraction of the population with newly generated nests.

In [ ]:
def levy_flight(beta: float = 1.5) -> float:
    """Draw a Mantegna Lévy-flight step for a valid stability parameter."""
    if not 0.0 < beta < 2.0:
        raise ValueError("The Lévy stability parameter beta must satisfy 0 < beta < 2.")

    sigma_u = (
        gamma(1.0 + beta)
        * math.sin(math.pi * beta / 2.0)
        / (
            gamma((1.0 + beta) / 2.0)
            * beta
            * 2.0 ** ((beta - 1.0) / 2.0)
        )
    ) ** (1.0 / beta)
    u = np.random.normal(0.0, sigma_u)
    v = np.random.normal(0.0, 1.0)
    while math.isclose(v, 0.0, abs_tol=1e-15):
        v = np.random.normal(0.0, 1.0)
    return float(u / (abs(v) ** (1.0 / beta)))


def cuckoo_egg_laying(current_nest: Nest, best_nest: Nest) -> Nest:
    """Create and evaluate one Lévy-flight-guided schedule candidate."""
    new_nest = copy.deepcopy(current_nest)
    vessel_id = random.choice([vessel.id for vessel in vessels])
    levy_step = levy_flight()

    if random.random() < 0.5:
        current_berth = new_nest.vessel_assignments[vessel_id]
        best_berth = best_nest.vessel_assignments[vessel_id]
        if abs(levy_step) > 1.0:
            new_berth = random.randrange(len(berths))
        elif random.random() < 0.5:
            new_berth = best_berth
        else:
            alternatives = [berth.id for berth in berths if berth.id != current_berth]
            new_berth = random.choice(alternatives) if alternatives else current_berth
        new_nest.vessel_assignments[vessel_id] = new_berth
    else:
        vessel = vessels[vessel_id]
        berth_id = new_nest.vessel_assignments[vessel_id]
        berth = berths[berth_id]
        feasible_times = [
            float(start_time)
            for start_time in np.arange(vessel.arrival_time, 24.0, 0.5)
            if is_feasible_assignment(vessel, berth, float(start_time))
        ]

        if feasible_times:
            current_time = new_nest.start_times[vessel_id]
            best_time = best_nest.start_times[vessel_id]
            if abs(levy_step) > 2.0:
                new_time = random.choice(feasible_times)
            else:
                reference_time = best_time if random.random() < 0.5 else current_time
                nearby_times = [
                    start_time
                    for start_time in feasible_times
                    if abs(start_time - reference_time) <= 2.0
                ]
                new_time = random.choice(nearby_times or feasible_times)
            new_nest.start_times[vessel_id] = new_time

    new_nest.feasible = is_nest_feasible(new_nest)
    new_nest.objective_value = calculate_objective(new_nest)
    new_nest.fitness = calculate_fitness(new_nest)
    return new_nest


def abandon_worst_nests(
    nests: List[Nest], abandonment_rate: float = 0.25
) -> List[Nest]:
    """Replace the weakest fraction of nests with newly generated schedules."""
    if not 0.0 <= abandonment_rate <= 1.0:
        raise ValueError("The abandonment rate must lie between 0 and 1.")

    num_to_abandon = int(len(nests) * abandonment_rate)
    if num_to_abandon == 0:
        return nests

    sorted_nests = sorted(nests, key=lambda nest: nest.fitness)
    for index in range(num_to_abandon):
        sorted_nests[index] = generate_random_nest()
    return sorted_nests


print("Lévy-flight and nest-management operators are ready.")

## 7. Cuckoo Search algorithm

Let \(N\) be the population size, \(T\) the number of iterations, and \(p_a\) the discovery probability. At each iteration, candidates are generated from current nests and compete with randomly selected nests. With probability \(p_a\), the weakest “fraction” of nests is replaced; the number replaced is the integer part of \(Np_a\). The best schedule found so far is retained separately, so abandonment cannot erase the recorded incumbent.

The population-based search performs roughly \(O(TN)\) candidate-generation and comparison steps, in addition to the work required to check each candidate. Here, schedule feasibility checking is proportional to the number of vessels, with sorting of berth schedules adding at most \(O(n\log n)\) for \(n\) vessels. This describes computational work, not a guarantee of solution quality.

In [ ]:
def cuckoo_search(
    population_size: int = 25,
    max_iterations: int = 1000,
    discovery_probability: float = 0.25,
    beta: float = 1.5,
    verbose: bool = True,
) -> Tuple[Nest, List[float]]:
    """Search for a low-cost feasible berth schedule using Cuckoo Search."""
    if population_size < 1:
        raise ValueError("Population size must be at least one.")
    if max_iterations < 0:
        raise ValueError("The iteration limit cannot be negative.")
    if not 0.0 <= discovery_probability <= 1.0:
        raise ValueError("Discovery probability must lie between 0 and 1.")
    if not 0.0 < beta < 2.0:
        raise ValueError("The Lévy stability parameter beta must satisfy 0 < beta < 2.")

    nests = [generate_random_nest() for _ in range(population_size)]
    best_nest = copy.deepcopy(max(nests, key=lambda nest: nest.fitness))
    convergence_history = [best_nest.objective_value]

    if verbose:
        print(f"Cuckoo Search initialized with {population_size} nests")
        print(f"Initial best objective: {best_nest.objective_value:.2f}")
        print(
            f"Discovery probability: {discovery_probability}, "
            f"Lévy stability parameter: {beta}"
        )
        print("=" * 60)

    for iteration in range(max_iterations):
        for index in range(population_size):
            new_nest = cuckoo_egg_laying(nests[index], best_nest)
            random_nest_index = random.randrange(population_size)

            if new_nest.fitness > nests[random_nest_index].fitness:
                nests[random_nest_index] = new_nest

            if new_nest.fitness > best_nest.fitness:
                best_nest = copy.deepcopy(new_nest)

        if random.random() < discovery_probability:
            nests = abandon_worst_nests(nests, discovery_probability)

        convergence_history.append(best_nest.objective_value)

        if verbose and (iteration % 100 == 0 or iteration == max_iterations - 1):
            feasible_count = sum(nest.feasible for nest in nests)
            average_fitness = float(np.mean([nest.fitness for nest in nests]))
            print(
                f"Iteration {iteration + 1:4d}: "
                f"Best={best_nest.objective_value:10.2f}, "
                f"Feasible={feasible_count}/{population_size}, "
                f"Average fitness={average_fitness:.8f}"
            )

    if verbose:
        print(f"\nFinal best objective: {best_nest.objective_value:.2f}")
        print(f"Feasible: {best_nest.feasible}")
        print(f"Iterations completed: {max_iterations}")

    return best_nest, convergence_history


print("Cuckoo Search is ready to run.")

## 8. Run the search

The parameters below define the main run. The random seeds set in the imports cell make the sequence of random choices reproducible when this notebook is run from the beginning in the same environment. The execution time is measured locally and is not a fixed property of the algorithm.

In [ ]:
start_time = time.time()
best_nest, convergence_history = cuckoo_search(
    population_size=25,
    max_iterations=1000,
    discovery_probability=0.25,
    beta=1.5,
    verbose=True,
)
execution_time = time.time() - start_time

print(f"\nExecution time: {execution_time:.2f} seconds")
print(f"Convergence-history length: {len(convergence_history)}")

## 9. Inspect the best schedule

The schedule report shows the assigned berth, service interval, elapsed time from arrival to completion, draft requirement, and the corresponding berth depth at service start. The final checks compare the reported objective with the objective function and verify feasibility. The search result is the best candidate found, not a certificate of global optimality.

In [ ]:
def analyze_nest(nest: Nest) -> None:
    """Print schedule details and verify the reported cost and feasibility."""
    print("Best-schedule analysis")
    print("=" * 72)

    if not nest.feasible:
        print("No feasible schedule was found.")
        return

    reported_total = 0.0
    for vessel_id in sorted(nest.vessel_assignments):
        vessel = vessels[vessel_id]
        berth_id = nest.vessel_assignments[vessel_id]
        berth = berths[berth_id]
        start_time = nest.start_times[vessel_id]
        end_time = start_time + vessel.processing_time
        elapsed_time = end_time - vessel.arrival_time
        cost = vessel.demurrage_cost * elapsed_time
        reported_total += cost
        available_depth = get_tidal_depth(start_time, berth)

        print(f"Vessel {vessel_id}:")
        print(f"  Berth: {berth_id}")
        print(f"  Service interval: {start_time:.1f} to {end_time:.1f} hours")
        print(f"  Elapsed time from arrival to completion: {elapsed_time:.1f} hours")
        print(f"  Draft: {vessel.draft:.1f} m; available start-time depth: {available_depth:.1f} m")
        print(f"  Weighted cost: {cost:,.0f}")

    print(f"\nReported total cost: {reported_total:,.2f}")
    print(f"Objective-function value: {nest.objective_value:,.2f}")
    print(f"Fitness: {nest.fitness:.10f}")
    print(f"Schedule feasible: {is_nest_feasible(nest)}")

    assert is_nest_feasible(nest), "The reported schedule must satisfy all constraints."
    assert math.isclose(
        reported_total, nest.objective_value, rel_tol=1e-12, abs_tol=1e-8
    ), "The reported cost must equal the objective-function value."


analyze_nest(best_nest)

## 10. Convergence behavior

The convergence history records the incumbent objective after initialization and after each iteration. Because the best-so-far schedule is retained, a finite incumbent objective should not increase during the run. If no feasible schedule is found, the objective remains infinite and percentage-improvement calculations are not meaningful; the plot function handles that case without reporting a fabricated statistic.

In [ ]:
def plot_convergence(convergence_history: List[float]) -> None:
    """Plot incumbent cost and summarize the observed search trajectory."""
    if not convergence_history:
        raise ValueError("The convergence history must contain at least one value.")

    iterations = np.arange(len(convergence_history))
    values = np.asarray(convergence_history, dtype=float)
    finite_values = np.isfinite(values)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    ax1, ax2, ax3, ax4 = axes.flat

    ax1.set_title("Cuckoo Search: Best-So-Far Objective")
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Objective cost (model units)")
    ax1.grid(True, alpha=0.3)
    if finite_values.any():
        ax1.plot(iterations[finite_values], values[finite_values], color="blue", label="Best objective")
        ax1.legend()
    else:
        ax1.text(0.5, 0.5, "No feasible schedule recorded", ha="center", va="center", transform=ax1.transAxes)

    ax2.set_title("Iteration-to-Iteration Improvement")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Relative improvement (%)")
    ax2.grid(True, alpha=0.3)
    if len(values) > 1:
        improvements = []
        improvement_iterations = []
        for index in range(1, len(values)):
            previous = values[index - 1]
            current = values[index]
            if math.isfinite(previous) and previous > 0 and math.isfinite(current):
                improvements.append((previous - current) / previous * 100.0)
                improvement_iterations.append(index)
        if improvements:
            ax2.plot(improvement_iterations, improvements, color="green", label="Relative improvement")
            ax2.axhline(0.0, color="red", linestyle="--", alpha=0.5)
            ax2.legend()

    ax3.set_title("Moving Average of Best-So-Far Objective (Window = 50)")
    ax3.set_xlabel("Iteration")
    ax3.set_ylabel("Objective cost (model units)")
    ax3.grid(True, alpha=0.3)
    window_size = 50
    if len(values) >= window_size and np.isfinite(values).all():
        moving_average = np.convolve(
            values, np.ones(window_size) / window_size, mode="valid"
        )
        ax3.plot(
            np.arange(window_size - 1, len(values)),
            moving_average,
            color="purple",
            label="Moving average",
        )
        ax3.legend()

    ax4.set_title("Search Trajectory by Iteration Segment")
    ax4.set_xlabel("Iteration")
    ax4.set_ylabel("Objective cost (model units)")
    ax4.grid(True, alpha=0.3)
    if finite_values.any():
        segments = np.array_split(np.arange(len(values)), 3)
        segment_names = ["Early", "Middle", "Late"]
        segment_colors = ["blue", "green", "red"]
        for segment, name, color in zip(segments, segment_names, segment_colors):
            valid_segment = segment[np.isfinite(values[segment])]
            if len(valid_segment):
                ax4.plot(
                    valid_segment,
                    values[valid_segment],
                    color=color,
                    label=f"{name} segment",
                )
        ax4.legend()

    plt.tight_layout()
    plt.show()

    if finite_values.any():
        initial = values[np.flatnonzero(finite_values)[0]]
        final = values[np.flatnonzero(finite_values)[-1]]
        print("Convergence summary:")
        print(f"  First finite objective: {initial:.2f}")
        print(f"  Final objective: {final:.2f}")
        if initial > 0:
            improvement = (initial - final) / initial * 100.0
            print(f"  Relative reduction from first finite value: {improvement:.2f}%")
        target = final + 0.1 * (initial - final)
        reached = next(
            (
                index
                for index, value in enumerate(values)
                if math.isfinite(value) and value <= target
            ),
            None,
        )
        if reached is None:
            print("  The 90%-of-observed-improvement threshold was not reached.")
        else:
            print(f"  First iteration at or below that threshold: {reached}")
    else:
        print("No finite objective was recorded, so convergence statistics are unavailable.")


plot_convergence(convergence_history)

## 11. Visualize the schedule and depth conditions

The upper panel is a berth-occupancy chart: each rectangle spans the vessel's service interval on its assigned berth. Shaded bands identify high-tide start-time windows, and dashed vertical lines mark vessel arrivals. The lower panel compares berth depth over time with each vessel's draft during service. Since the model checks draft at the service start, that panel displays the full service interval for context; it should not be read as an additional full-service tidal-depth constraint.

In [ ]:
def visualize_nest(nest: Nest) -> None:
    """Plot berth occupancy and the model's start-time depth conditions."""
    if not nest.feasible:
        print("A schedule plot is unavailable because the candidate is infeasible.")
        return

    vessel_colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

    ax1.set_title("Cuckoo Search Schedule: Berth Occupancy")
    ax1.set_xlabel("Time (hours)")
    ax1.set_ylabel("Berth")
    ax1.set_xlim(0, 24)
    ax1.set_ylim(-0.5, len(berths) - 0.5)
    ax1.set_yticks(range(len(berths)))
    ax1.set_yticklabels([f"Berth {berth.id}" for berth in berths])
    ax1.grid(True, alpha=0.3)

    for index, window in enumerate(tidal_windows):
        ax1.axvspan(
            window.start_time,
            window.end_time,
            alpha=0.2,
            color="lightblue",
            label="High-tide start window" if index == 0 else None,
        )

    for vessel_id in sorted(nest.vessel_assignments):
        berth_id = nest.vessel_assignments[vessel_id]
        start_time = nest.start_times[vessel_id]
        end_time = start_time + vessels[vessel_id].processing_time
        color = vessel_colors[vessel_id % len(vessel_colors)]
        rectangle = patches.Rectangle(
            (start_time, berth_id - 0.3),
            end_time - start_time,
            0.6,
            linewidth=1.5,
            edgecolor="black",
            facecolor=color,
            alpha=0.8,
            label=f"Vessel {vessel_id}",
        )
        ax1.add_patch(rectangle)
        ax1.text(
            (start_time + end_time) / 2,
            berth_id,
            f"V{vessel_id}",
            ha="center",
            va="center",
            fontweight="bold",
        )

    for index, vessel in enumerate(vessels):
        ax1.axvline(
            vessel.arrival_time,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Vessel arrival" if index == 0 else None,
        )
    ax1.legend(loc="upper right", ncol=2)

    ax2.set_title("Berth Depth and Vessel Draft")
    ax2.set_xlabel("Time (hours)")
    ax2.set_ylabel("Depth (m)")
    ax2.set_xlim(0, 24)
    ax2.grid(True, alpha=0.3)

    time_points = np.linspace(0, 24, 481)
    for berth in berths:
        depths = [get_tidal_depth(float(time_point), berth) for time_point in time_points]
        ax2.plot(time_points, depths, linewidth=2, label=f"Berth {berth.id} available depth")

    for vessel_id in sorted(nest.vessel_assignments):
        vessel = vessels[vessel_id]
        start_time = nest.start_times[vessel_id]
        end_time = start_time + vessel.processing_time
        color = vessel_colors[vessel_id % len(vessel_colors)]
        ax2.hlines(
            vessel.draft,
            start_time,
            end_time,
            colors=color,
            linewidth=3,
            label=f"Vessel {vessel_id} draft",
        )
        ax2.axvspan(start_time, end_time, alpha=0.08, color=color)

    ax2.legend(loc="best", ncol=2)
    plt.tight_layout()
    plt.show()


visualize_nest(best_nest)

## 12. Parameter sensitivity

A stochastic search can respond to population size, discovery probability, and the Lévy stability parameter. The experiment below changes one parameter group at a time while holding the other settings fixed. Each setting is evaluated in one run, so the results are observations for this seeded experiment, not statistical guarantees or universal parameter recommendations. In particular, \(0<\beta<2\) is required by the Lévy-flight construction.

In [ ]:
def parameter_sensitivity_analysis() -> Dict[str, List[Dict[str, float]]]:
    """Record one-run outcomes for selected Cuckoo Search parameters."""
    results: Dict[str, List[Dict[str, float]]] = {}

    experiments = [
        (
            "Population size",
            "population_size",
            [10, 25, 50, 75],
            {"max_iterations": 200, "discovery_probability": 0.25, "beta": 1.5},
        ),
        (
            "Discovery probability",
            "discovery_probability",
            [0.1, 0.25, 0.4, 0.5],
            {"population_size": 25, "max_iterations": 200, "beta": 1.5},
        ),
        (
            "Lévy stability parameter",
            "beta",
            [1.0, 1.5, 1.9],
            {"population_size": 25, "max_iterations": 200, "discovery_probability": 0.25},
        ),
    ]

    for title, parameter_name, values, fixed_parameters in experiments:
        print(f"\n{title} sensitivity:")
        group_results = []
        for value in values:
            parameters = dict(fixed_parameters)
            parameters[parameter_name] = value
            start = time.time()
            nest, _ = cuckoo_search(verbose=False, **parameters)
            elapsed = time.time() - start
            record = {
                "parameter_value": value,
                "objective": nest.objective_value,
                "feasible": nest.feasible,
                "elapsed_seconds": elapsed,
            }
            group_results.append(record)
            objective_text = (
                f"{nest.objective_value:,.2f}" if nest.feasible else "unavailable"
            )
            print(
                f"  {parameter_name}={value}: objective={objective_text}, "
                f"feasible={nest.feasible}, elapsed={elapsed:.2f} s"
            )
        results[title] = group_results

    print(
        "\nInterpret each row as one stochastic run. Repeated trials would be "
        "needed to estimate variability reliably."
    )
    return results


sensitivity_results = parameter_sensitivity_analysis()

## 13. Repeatability across Cuckoo Search runs

Because the algorithm uses random initialization and random candidate changes, a single run cannot describe its run-to-run variability. This experiment repeats the same Cuckoo Search settings with different random streams and summarizes the feasible objective values and feasibility rate. The seed sequence is fixed to make the experiment reproducible. This is a repeatability analysis of the same method, not a comparison with another optimization algorithm.

In [ ]:
def repeatability_analysis(num_trials: int = 10) -> List[Dict[str, float]]:
    """Summarize variability across repeated runs of the same search method."""
    if num_trials < 1:
        raise ValueError("The number of trials must be at least one.")

    python_rng_state = random.getstate()
    numpy_rng_state = np.random.get_state()
    trial_results = []

    try:
        for trial in range(num_trials):
            seed = 1000 + trial
            random.seed(seed)
            np.random.seed(seed)
            start = time.time()
            nest, _ = cuckoo_search(
                population_size=25,
                max_iterations=200,
                discovery_probability=0.25,
                beta=1.5,
                verbose=False,
            )
            trial_results.append(
                {
                    "trial": trial + 1,
                    "objective": nest.objective_value,
                    "feasible": nest.feasible,
                    "elapsed_seconds": time.time() - start,
                }
            )
    finally:
        random.setstate(python_rng_state)
        np.random.set_state(numpy_rng_state)

    feasible_objectives = [
        result["objective"] for result in trial_results if result["feasible"]
    ]
    feasibility_rate = sum(result["feasible"] for result in trial_results) / num_trials
    print(f"Feasibility rate: {feasibility_rate:.1%} ({len(feasible_objectives)}/{num_trials})")

    if feasible_objectives:
        print(f"Best feasible objective: {min(feasible_objectives):,.2f}")
        print(f"Mean feasible objective: {np.mean(feasible_objectives):,.2f}")
        print(f"Standard deviation: {np.std(feasible_objectives):,.2f}")
    else:
        print("No feasible objective values were recorded.")

    print(f"Mean elapsed time: {np.mean([item['elapsed_seconds'] for item in trial_results]):.2f} s")
    return trial_results


repeatability_results = repeatability_analysis()

## 14. Interpretation and scope

Cuckoo Search represents a schedule as a nest, uses Lévy-flight-guided changes to explore alternative vessel assignments and start times, compares candidate fitness, and periodically replaces weak nests. The objective translates completion delay into vessel-weighted cost, while explicit feasibility checks enforce the model's arrival, tidal-start, draft, and berth-capacity requirements.

A lower feasible objective is preferable within this model. A candidate with no feasible schedule is not a valid operational plan, even if its numerical fitness is available for population management. The convergence curve describes the best candidate observed during one run; it does not establish global optimality. Sensitivity and repeatability results are empirical and depend on the instance, random seeds, and computational environment.

This example uses a small fixed instance and a half-hour start-time grid. It does not model continuous-time optimization, vessel lengths, spatial berth positions, tide evolution within service, uncertainty, or operational rules not stated in the assumptions. These are possible extensions of the same berth-allocation model; they are not implemented here.